In [1]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [2]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

# --- Count stats in the training set ---
count_mean  = count_stats[0]
count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.01 | Test MAE: 79.18 bikes/hour


In [3]:
count_norm = (counts - count_mean) / count_std

In [4]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Índices de inicio válidos: cada secuencia de longitud
        # sequence_length, muestreada cada sampling_rate pasos, abarca
        # `span` filas, y el objetivo queda `delay` filas después del
        # inicio. Acotamos end_index para que ninguna ventana se salga
        # del final del arreglo.
        span      = (sequence_length - 1) * sampling_rate
        max_start = len(data) - span - delay - 1
        safe_end  = min(end_index, max_start + 1)
        self.indices = np.arange(start_index, safe_end)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Muestrea cada `sampling_rate` pasos durante `sequence_length` pasos
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # El objetivo está `delay` pasos adelante del inicio de la secuencia
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [5]:
sampling_rate   = 1    # ya es horario
sequence_length = 168  # mira 168 horas hacia atrás = 1 semana completa
delay           = 1    # predice 1 hora adelante
batch_size      = 256

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Verificación rápida ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 168, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 168, 28])
Target shape: torch.Size([256])
Target range: -1 to 4


## 1. Modelo — GRU

Empezamos simple, tal como recomiendan los tips: **una sola capa GRU con 64
unidades ocultas** seguida de una capa lineal que mapea el último estado
oculto a una única salida de regresión (el conteo de bicis normalizado una
hora adelante).

- La GRU consume la secuencia completa `(batch, seq_len, num_features)` y
  devuelve una salida por cada paso de tiempo. Nos quedamos solo con el
  **estado oculto del último paso**, que resume toda la ventana, y lo pasamos
  a la capa lineal.
- `num_layers`, `dropout` y `bidirectional` se exponen como argumentos para
  poder hacer el modelo más complejo después *solo si* la brecha train/val
  muestra que la versión simple sufre de underfitting u overfitting. El
  dropout se aplica entre capas GRU apiladas (PyTorch solo lo activa cuando
  `num_layers > 1`) y de nuevo antes de la capa lineal.

In [6]:
class GRURegressor(nn.Module):
    """Regresor de una sola salida construido sobre un encoder GRU.

    Args:
        input_size:    número de features de entrada por paso (28 aquí).
        hidden_size:   unidades ocultas de la GRU (empezar con 64).
        num_layers:    número de capas GRU apiladas (empezar con 1).
        dropout:       prob. de dropout entre capas y antes de la salida.
        bidirectional: si se lee la secuencia en ambas direcciones.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=1,
                 dropout=0.0, bidirectional=False):
        super().__init__()
        self.num_layers    = num_layers
        self.hidden_size   = hidden_size
        self.bidirectional = bidirectional

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,                 # entrada es (batch, seq, feature)
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        num_directions = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(hidden_size * num_directions, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, h_n = self.gru(x)              # out: (batch, seq_len, hidden*dirs)

        if self.bidirectional:
            # concatena el último estado hacia adelante y hacia atrás
            h_fwd = h_n[-2]
            h_bwd = h_n[-1]
            last = torch.cat([h_fwd, h_bwd], dim=1)
        else:
            last = h_n[-1]                  # (batch, hidden) — estado capa superior

        last = self.dropout(last)
        y = self.head(last)                # (batch, 1)
        return y.squeeze(1)                # (batch,) para igualar la forma del objetivo

In [ ]:
# --- Dispositivo + instanciación del modelo ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_features = raw_data.shape[1]           # 28

# Partimos del modelo simple (1 capa / 64 unidades), que ya le gana al
model = GRURegressor(
    input_size=num_features,
    hidden_size=128,
    num_layers=2,
    dropout=0.4,
    bidirectional=True,
).to(device)

print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

Using device: cuda
GRURegressor(
  (gru): GRU(28, 128, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (head): Linear(in_features=256, out_features=1, bias=True)
)
Trainable parameters: 418,049


## 2. Configuración del entrenamiento

- **Loss:** MSE sobre los objetivos *normalizados*. Entrenar en la escala
  normalizada mantiene los gradientes bien condicionados; convertimos de
  vuelta a bicis/hora solo al reportar el MAE.
- **Optimizador:** Adam, `lr=1e-3` (bajar a `1e-4` si el loss se estanca pronto).
- **ReduceLROnPlateau:** reduce el LR a la mitad cuando el MAE de validación
  deja de mejorar, así no hay que ajustar el schedule a mano.
- **ModelCheckpoint (manual):** cada vez que el MAE de val alcanza un nuevo
  mejor valor, guardamos los pesos en `best_model.pt`. El entrenamiento puede
  correr largo sin problema — nunca se pierde el mejor punto.
- **Early stopping (manual):** si el MAE de val no mejora durante `patience`
  epochs, nos detenemos.
- **TensorBoard:** el loss y el MAE de train/val se registran en `runs/` para
  ver las curvas en vivo con `tensorboard --logdir runs`.

Como los objetivos están normalizados, **MAE en bicis/hora = MAE_normalizado ×
`count_std`** (el desplazamiento de la media se cancela dentro de una
diferencia absoluta). Esa es la única conversión usada en todo lo que sigue.

In [8]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import datetime

# --- Hiperparámetros ---
num_epochs    = 150     # corre largo; el early stopping decide cuándo parar
learning_rate = 1e-3
patience      = 20      # paciencia del early stopping (epochs sin mejora en val)
ckpt_path     = "best_model.pt"

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

# Writer de TensorBoard con un nombre de run único por ejecución
run_name = datetime.datetime.now().strftime("gru_%Y%m%d-%H%M%S")
writer   = SummaryWriter(os.path.join("runs", run_name))
print(f"Logging to runs/{run_name}")

Logging to runs/gru_20261022-232930


In [9]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Devuelve (loss MSE promedio, MAE en bicis/hora) sobre un loader."""
    model.eval()
    total_loss = 0.0
    total_abs  = 0.0        # suma de |error| en unidades normalizadas
    n_samples  = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_abs  += torch.sum(torch.abs(pred - y)).item()
        n_samples  += bs

    mean_loss = total_loss / n_samples
    mae_norm  = total_abs / n_samples
    mae_bikes = mae_norm * count_std        # convierte a bicis/hora
    return mean_loss, mae_bikes

In [10]:
# --- Loop de entrenamiento ---
best_val_mae = float("inf")
epochs_no_improve = 0

for epoch in range(1, num_epochs + 1):
    # ---- entrenamiento ----
    model.train()
    running_loss = 0.0
    running_abs  = 0.0
    n_seen       = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        running_loss += loss.item() * bs
        running_abs  += torch.sum(torch.abs(pred.detach() - y)).item()
        n_seen       += bs

    train_loss = running_loss / n_seen
    train_mae  = (running_abs / n_seen) * count_std

    # ---- validación ----
    val_loss, val_mae = evaluate(model, val_loader, criterion, device)

    # ---- scheduler según el MAE de val ----
    scheduler.step(val_mae)
    current_lr = optimizer.param_groups[0]["lr"]

    # ---- registro en TensorBoard ----
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val",   val_loss,   epoch)
    writer.add_scalar("MAE/train",  train_mae,  epoch)
    writer.add_scalar("MAE/val",    val_mae,    epoch)
    writer.add_scalar("LR",         current_lr, epoch)

    # ---- guarda el mejor modelo (checkpoint) ----
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        epochs_no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        flag = "  <-- best, saved"
    else:
        epochs_no_improve += 1
        flag = ""

    print(f"Epoch {epoch:3d}/{num_epochs} | "
          f"train MAE {train_mae:6.2f} | val MAE {val_mae:6.2f} | "
          f"lr {current_lr:.1e}{flag}")

    # ---- parada temprana (early stopping) ----
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch} "
              f"(no val improvement for {patience} epochs).")
        break

writer.close()
print(f"\nBest val MAE: {best_val_mae:.2f} bikes/hour  (weights in {ckpt_path})")

Epoch   1/150 | train MAE  75.08 | val MAE  97.95 | lr 1.0e-03  <-- best, saved


Epoch   2/150 | train MAE  54.57 | val MAE  74.47 | lr 1.0e-03  <-- best, saved


Epoch   3/150 | train MAE  42.00 | val MAE  58.06 | lr 1.0e-03  <-- best, saved


Epoch   4/150 | train MAE  36.08 | val MAE  48.47 | lr 1.0e-03  <-- best, saved


Epoch   5/150 | train MAE  33.32 | val MAE  47.63 | lr 1.0e-03  <-- best, saved


...

Epoch  29/150 | train MAE  21.86 | val MAE  51.35 | lr 2.5e-04


Epoch  30/150 | train MAE  21.88 | val MAE  51.89 | lr 2.5e-04


Epoch  31/150 | train MAE  21.75 | val MAE  49.38 | lr 1.3e-04


Epoch  32/150 | train MAE  21.25 | val MAE  52.58 | lr 1.3e-04


Epoch  33/150 | train MAE  21.27 | val MAE  51.35 | lr 1.3e-04

Early stopping at epoch 33 (no val improvement for 20 epochs).

Best val MAE: 43.04 bikes/hour  (weights in best_model.pt)


## 3. Evaluación

Recargamos el **mejor** checkpoint (no el último epoch, que puede haber hecho
overfitting) y reportamos el MAE en bicis/hora sobre validación y test.
Comparamos contra:

- el **baseline naive** cargado al inicio, y
- la meta de quedar **dentro de 10 bicis/hora del baseline de sklearn
  (< 44 bicis/hora)**, que los tips llaman un resultado excelente.

In [11]:
# --- Carga el mejor checkpoint y evalúa ---
model.load_state_dict(torch.load(ckpt_path, map_location=device))

val_loss,  val_mae  = evaluate(model, val_loader,  criterion, device)
test_loss, test_mae = evaluate(model, test_loader, criterion, device)

print("=" * 52)
print(f"{'Modelo':<22}{'MAE Val':>14}{'MAE Test':>14}")
print("-" * 52)
print(f"{'Baseline naive':<22}{naive_mae[0]:>11.2f}   {naive_mae[1]:>11.2f}")
print(f"{'GRU (nuestro)':<22}{val_mae:>11.2f}   {test_mae:>11.2f}")
print("=" * 52)

print(f"\nMejora sobre el naive (test): "
      f"{naive_mae[1] - test_mae:+.2f} bicis/hora")
if test_mae < 44:
    print("MAE test < 44 bicis/hora — resultado excelente (a menos de 10 del de sklearn).")
else:
    print("MAE test >= 44 — prueba más epochs, ajusta el LR o agrega regularización ligera.")

Modelo                       MAE Val      MAE Test
----------------------------------------------------
Baseline naive              93.01         79.18
GRU (nuestro)               43.04         48.41

Mejora sobre el naive (test): +30.77 bicis/hora
MAE test >= 44 — prueba más epochs, ajusta el LR o agrega regularización ligera.
